#### Black–Scholes

Runnable companion for [`black-scholes.md`](./black-scholes.md).

1. Closed-form European call/put with Greeks.
2. Numerical put–call parity check.
3. Call price vs strike across several volatilities; Delta vs spot.
4. Monte-Carlo price under GBM vs closed-form.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(0)

In [ ]:
def bs_d1_d2(S, K, r, sigma, T):
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return d1, d2

def bs_price(S, K, r, sigma, T, kind='call'):
    d1, d2 = bs_d1_d2(S, K, r, sigma, T)
    if kind == 'call':
        return S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
    return K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)

def bs_greeks(S, K, r, sigma, T, kind='call'):
    d1, d2 = bs_d1_d2(S, K, r, sigma, T)
    pdf = norm.pdf(d1)
    delta = norm.cdf(d1) if kind == 'call' else norm.cdf(d1) - 1
    gamma = pdf / (S * sigma * np.sqrt(T))
    vega = S * pdf * np.sqrt(T)
    if kind == 'call':
        theta = -S * pdf * sigma / (2 * np.sqrt(T)) - r * K * np.exp(-r * T) * norm.cdf(d2)
        rho = K * T * np.exp(-r * T) * norm.cdf(d2)
    else:
        theta = -S * pdf * sigma / (2 * np.sqrt(T)) + r * K * np.exp(-r * T) * norm.cdf(-d2)
        rho = -K * T * np.exp(-r * T) * norm.cdf(-d2)
    return dict(delta=delta, gamma=gamma, vega=vega, theta=theta, rho=rho)

S0, K, r, sigma, T = 100.0, 100.0, 0.05, 0.20, 1.0
C = bs_price(S0, K, r, sigma, T, 'call')
P = bs_price(S0, K, r, sigma, T, 'put')
print(f'Call = {C:.4f}   Put = {P:.4f}')
print('Greeks (call):', {k: round(float(v), 4) for k, v in bs_greeks(S0, K, r, sigma, T, 'call').items()})

#### Put–call parity check

$C - P = S_0 - K e^{-rT}$ should hold for any valid pricer.

In [ ]:
lhs = C - P
rhs = S0 - K * np.exp(-r * T)
print(f'C - P              = {lhs:.6f}')
print(f'S0 - K * exp(-rT)  = {rhs:.6f}')
print(f'|lhs - rhs|        = {abs(lhs - rhs):.2e}')

# Sweep strikes — parity must hold pointwise
Ks = np.linspace(60, 140, 9)
calls = bs_price(S0, Ks, r, sigma, T, 'call')
puts  = bs_price(S0, Ks, r, sigma, T, 'put')
gap   = (calls - puts) - (S0 - Ks * np.exp(-r * T))
print('\nmax parity violation across strikes:', np.max(np.abs(gap)))

#### Call price vs strike at several volatilities, and Delta vs spot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

Ks = np.linspace(50, 150, 200)
for s in [0.10, 0.20, 0.40, 0.60]:
    axes[0].plot(Ks, bs_price(S0, Ks, r, s, T, 'call'), label=f'sigma={s:.2f}')
axes[0].plot(Ks, np.maximum(S0 - Ks * np.exp(-r * T), 0), 'k--', lw=1, label='intrinsic')
axes[0].set_xlabel('strike K'); axes[0].set_ylabel('call price')
axes[0].set_title('Call price vs strike'); axes[0].legend()

Ss = np.linspace(40, 160, 200)
for tau in [0.05, 0.25, 1.0, 2.0]:
    deltas = bs_greeks(Ss, K, r, sigma, tau, 'call')['delta']
    axes[1].plot(Ss, deltas, label=f'T={tau:.2f}')
axes[1].axvline(K, color='gray', lw=0.5)
axes[1].set_xlabel('spot S'); axes[1].set_ylabel('Delta')
axes[1].set_title('Call Delta vs spot (K=100)'); axes[1].legend()

plt.tight_layout(); plt.show()

#### Monte-Carlo price under GBM

Simulate $S_T = S_0 \exp((r - \tfrac12 \sigma^2) T + \sigma \sqrt{T} Z)$ with $Z \sim \mathcal{N}(0, 1)$, take the discounted average payoff, and compare to the closed form.

In [ ]:
M = 200_000
Z = rng.standard_normal(M)
ST = S0 * np.exp((r - 0.5 * sigma**2) * T + sigma * np.sqrt(T) * Z)

call_payoff = np.maximum(ST - K, 0.0)
put_payoff  = np.maximum(K - ST, 0.0)
disc = np.exp(-r * T)

mc_call = disc * call_payoff.mean()
se_call = disc * call_payoff.std(ddof=1) / np.sqrt(M)
mc_put  = disc * put_payoff.mean()
se_put  = disc * put_payoff.std(ddof=1) / np.sqrt(M)

print(f'Closed-form call = {C:.4f}')
print(f'Monte-Carlo call = {mc_call:.4f}  (SE {se_call:.4f}, {abs(mc_call-C)/se_call:.2f} SE away)')
print(f'Closed-form put  = {P:.4f}')
print(f'Monte-Carlo put  = {mc_put:.4f}  (SE {se_put:.4f}, {abs(mc_put-P)/se_put:.2f} SE away)')

In [ ]:
# Convergence: error vs sample size
sizes = np.logspace(2, 5.5, 12).astype(int)
errs, ses = [], []
for n in sizes:
    Zn = rng.standard_normal(n)
    STn = S0 * np.exp((r - 0.5 * sigma**2) * T + sigma * np.sqrt(T) * Zn)
    payoff = np.maximum(STn - K, 0.0)
    est = disc * payoff.mean()
    errs.append(abs(est - C))
    ses.append(disc * payoff.std(ddof=1) / np.sqrt(n))

fig, ax = plt.subplots(figsize=(7, 4))
ax.loglog(sizes, errs, 'o-', label='|MC - closed form|')
ax.loglog(sizes, ses, 's--', label='1 standard error')
ax.loglog(sizes, errs[0] * np.sqrt(sizes[0] / sizes), 'k:', label='~1/sqrt(N) reference')
ax.set_xlabel('number of MC samples'); ax.set_ylabel('absolute error')
ax.set_title('Monte-Carlo convergence to Black-Scholes call price')
ax.legend(); ax.grid(True, which='both', alpha=0.3)
plt.show()